&nbsp;

# Bulk RNA-seq Metadata Final Check

- 可視化などに必要な情報を整備するために実行する
- サンプルのオントロジーへのマッピングなどを行ったバージョンを figshare に掲載する

&nbsp;

# Assign Ontology Process

- ENCODE-rE2G の メタデータから、サンプルのオントロジーのマッピング情報があり、参考にできると考え実行
- この notebook は、メタデータの最終チェックも兼ねている

In [1]:
import datetime

import polars as pl
import pathlib
from IPython.display import display

pl.Config.set_fmt_str_lengths(200)
pl.Config.set_tbl_cols(200)
pl.Config.set_tbl_rows(4)

polars.config.Config

&nbsp;

&nbsp;

## Reference: ENCODE-rE2G metadata

- メタデータに、細胞株とオントロジーへのマッピング結果が書いてあるので参考にする

In [2]:
encode_re2g_metadata = pl.read_csv(
    "../../out/ENCODE_rE2G/ctrl_list/metadata/ENCODE-rE2G_bedpe_map_supplementary_table_12_join.tsv",
    separator="\t",
    has_header=True
)

display(encode_re2g_metadata)

ENCODE-rE2G accession,predicted_model,files_annotation_type,files_accession,files_file_type,files_derived_from,files_assembly,files_biosample_ontology,files_assay_term_name,files_date_created,files_output_type,files_content_md5sum,files_md5sum,Biosample term id,Model,Sample,Biosample term name,Biosample term id_table12,Biosample type,Biosample organism,Biosample treatments,Biosample treatments amount,Biosample treatments duration,Biosample genetic modifications methods,Biosample genetic modifications categories,Biosample genetic modifications targets,Biosample genetic modifications gene targets,Biosample genetic modifications site coordinates,Biosample genetic modifications zygosity,DNase Experiment accession,DNase Experiment date released,DNase Lab,DNase Donor(s),DNase File accession,DNase Output type,DNase Run type,ENCODE-rE2G v1,ENCODE Hi-C Experiment accession,H3K27ac Experiment accession,H3K4me3 Experiment accession,CAGE Experiment accession,EP300 Experiment accession,ChIA-PET Experiment accession,CTCF Experiment accession,in situ Hi-C Experiment accession,term_id_mismatch
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,bool,str,str,str,str,str,str,str,str,bool
"""ENCSR997TVR""","""ENCODE-rE2G""","""element gene regulatory interaction predictions""","""ENCFF547ARY""","""bedpe""","""ENCFF461QVU""","""GRCh38""","""/biosample-types/cell_line_EFO_0002784/""","""DNase-seq, ChIP-seq, CAGE, HiC""","""2025-01-24T01:49:44.372281+00:00""","""thresholded links""","""0e4355f4bd057fc7aa2fc3a502d94ce4""","""bc1f0f0cf05fe8194cf9ea143f9d9031""","""EFO:0002784""","""Extended""","""GM12878_ENCSR000EMT""","""GM12878""","""EFO:0002784""","""cell line""","""Homo sapiens""",null,null,null,null,null,null,null,null,null,"""ENCSR000EMT""","""5/4/11""","""John Stamatoyannopoulos, UW""","""/human-donors/ENCDO000AAK/""","""ENCFF467CXY, ENCFF940NSD""","""unfiltered alignments, unfiltered alignments""","""single-ended, single-ended""",true,"""ENCSR916MFV""","""ENCSR000AKC""","""ENCSR057BWO""","""ENCSR000CKA""","""ENCSR000DZD""","""ENCSR184YZV""","""ENCSR000DZN""","""ENCSR410MDC""",false
"""ENCSR621BIJ""","""ENCODE-rE2G""","""element gene regulatory interaction predictions""","""ENCFF647VTW""","""bedpe""","""ENCFF658KQS""","""GRCh38""","""/biosample-types/cell_line_EFO_0002067/""","""DNase-seq, ChIP-seq, CAGE, HiC""","""2025-01-24T01:49:37.854804+00:00""","""thresholded links""","""217fd3f386aad1674b22d612b6227241""","""f380fd6e8c9f452d2fe12315c03d4a11""","""EFO:0002067""","""Extended""","""K562_ENCSR000EOT""","""K562""","""EFO:0002067""","""cell line""","""Homo sapiens""",null,null,null,null,null,null,null,null,null,"""ENCSR000EOT""","""5/9/18""","""John Stamatoyannopoulos, UW""","""/human-donors/ENCDO000AAD/""","""ENCFF325RTP, ENCFF860XAE""","""unfiltered alignments, unfiltered alignments""","""paired-ended & single-ended, single-ended""",true,"""ENCSR479XDG""","""ENCSR000AKP""","""ENCSR668LDD""","""ENCSR000CJN""","""ENCSR000EGE""","""ENCSR597AKG""","""ENCSR000DWE""","""ENCSR545YBD""",false
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENCSR123AKF""","""ENCODE-rE2G""","""element gene regulatory interaction predictions""","""ENCFF580GKK""","""bedpe""","""ENCFF075XMV""","""GRCh38""","""/biosample-types/tissue_UBERON_0000996/""","""DNase-seq""","""2024-06-06T21:25:06.423949+00:00""","""thresholded links""","""a73717c936534f843d9670bebfd5f91a""","""981e371dd3c617f7ca1c35ad20eb3321""","""UBERON:0000996""","""DNase-only""","""vagina_ENCSR733YNW""","""vagina""","""UBERON:0000996""","""tissue""","""Homo sapiens""",null,null,null,null,null,null,null,null,null,"""ENCSR733YNW""","""11/1/17""","""John Stamatoyannopoulos, UW""","""/human-donors/ENCDO271OUW/""","""ENCFF564NGR""","""alignments""","""paired-ended""",true,"""ENCSR492BKE""",null,null,null,null,null,null,null,false
"""ENCSR738AUI""","""ENCODE-rE2G""","""elemen

In [3]:
encode_re2g_metadata_sample_list = encode_re2g_metadata.select(
    "Biosample term name",
    "Biosample term id"
).unique(
).sort(
    by="Biosample term name",
    descending=False
)

pl.Config.set_tbl_rows(2)
display(encode_re2g_metadata_sample_list)

Biosample term name,Biosample term id
str,str
"""8988T""","""EFO:0005713"""
…,…
"""vagina""","""UBERON:0000996"""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 1. Heat stress-related RNA-seq metadata

- もしサンプルなどに関して間違いがあれば、それもこの notebook で修正する
- 間違い1: PRJNA607507 stored the disease name in sample. The paper sequences parental DLD-1 cells.
- 間違い2: PRJNA915030 stored the cell type "T lymphocyte" and the DOI of a different paper. GEO GSE221579 sequences Jurkat T cells (doi:10.3390/ijms24021351).
- 間違い3: PRJEB44754 stored the culture category "primary cell line". The supplementary methods isolate human mesenchymal stem cells from bone marrow.
- 間違い4: DOI のミス

In [4]:
metadata_rnaseq_all = pl.read_csv(
    "../../Data/metadata/metadata_2610/rna_seq_all_metadata_simple_join_salmon_all_columns_261001.tsv",
    separator="\t",
    has_header=True,
    schema_overrides={
        "sample_contact_phone": pl.Utf8,
    }
)


metadata_rnaseq_all_sample_list = metadata_rnaseq_all.select(
    "study_accession",
    "reference_doi",
    "sample"
).unique().sort(
    by="sample",
    descending=False
)

pl.Config.set_tbl_rows(2)
display(metadata_rnaseq_all)

sample_heat_id,sample_control_id,sample_name,sample_title,experiment_accession,experiment_alias,secondary_sample_accession,study_accession,first_public,study_alias,instrument_platform,instrument_model,library_source,library_layout,sample_molecule_ch1,sample,stress treatment temperature,stress treatment time,control treatment temperature,rna_library,normalized_label,item_type,reference_doi,strandness,strandness_evidence,expected_format_heat,strand_mapping_bias_heat,expected_format_control,strand_mapping_bias_control,format_pair_comparison,expected_library_strandedness,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,cell_type,genotype,treatment,mapping,expression_estimation,deg_analysis,assembly,supplementary_files_format_and_content,sample_accession,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,country,ncbi_reporting_standard,sample_description_right,secondary_study_accession,read_count,study_title,sample_title_right,description,cell_type_right,base_count,library_strategy,status,sample_alias,run_alias,library_construction_protocol,collection_date,cell_line_right,library_name,first_created,sample_data_processing,sample_contact_email,tissue,biological_replicates,tissue_type,protocol,organism,read_type,data_source,srr,srx,sample_contact_department,passage,phenotype,replicate,genome_build,center_name,sample_contact_laboratory,sample_supplementary_file_2,sample_supplementary_file_3,disease,stress,sample_contact_phone,tumor,sex,age,sample_treatment_protocol_ch1,sample_group,cell_model,sample_extract_protocol_ch1,time,library_strategy_right,sample_growth_protocol_ch1,sirna,condition,tissue_cells,antibody,batch,column,library_name_right,sample_characteristics_ch1,centrifuge,strain,fraction,enzyme_chemical_treatment,rt_condition,strain_right,treatment_conditions,molecule,isolate,isolate_right,dosage,experiment,timepoint,vector,sirna_sgrna,sra_file_role,fastq_file_role,tag,scientific_name,tax_id,accession,collection_date_end,collection_date_start,submitted_galaxy,submitted_ftp,submitted_aspera,submitted_md5,checklist,submitted_bytes,nominal_sdev,broker_name,datahub,nominal_length,submitted_read_type,submitted_format,bam_ftp,bam_bytes,dev_stage,bam_aspera,bam_galaxy,bam_file_role,submitted_file_role,recovery treatment temperature,recovery treatment time,sample_series_id,biosample,sra,fastq_galaxy,fastq_ftp,fastq_aspera,tax_lineage,Read length for mate1(bp),Read length for mate 2(bp),Individual Accession,Biomaterial Provider
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,str,f64,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR27681984""","""SRR27681986""","""wi-38-htert_40°c_rna-seq_rep1 vs wi-38-htert_37°c_rna-seq_rep1""","""WI-38-hTERT, 40°C, RNA-seq, rep1 vs WI-38-hTERT, 37°C, RNA-seq, rep1""","""SRX23349232 vs SRX23349230""","""GSM8029688_r1

&nbsp;

&nbsp;

## Fix Metadata

- `Colorectal adenocarcinoma` を サンプルとして扱っていたが、DLD-1 cell だったことが判明し、修正
- `T lymphocyte` と Nature 論文の DOI が入っていたが、GEO の掲載論文は Jurkat T 細胞の過重力実験だったので、sample と reference_doi を修正
- `primary cell line` と入っていたが、補足 Method では骨髄由来の human mesenchymal stem cells だったので、sample を修正

In [5]:
# PRJNA607507 stored the disease name in sample. The paper sequences parental DLD-1 cells.
metadata_rnaseq_all = metadata_rnaseq_all.with_columns(
    pl.when(
        (pl.col("study_accession") == "PRJNA607507")
        & (pl.col("sample") == "Colorectal adenocarcinoma")
    )
    .then(pl.lit("DLD-1"))
    .otherwise(pl.col("sample"))
    .alias("sample")
)

metadata_rnaseq_all_sample_list = metadata_rnaseq_all.select(
    "study_accession",
    "reference_doi",
    "sample"
).unique().sort(
    by="sample",
    descending=False
)

display(
    metadata_rnaseq_all.filter(
        pl.col("study_accession") == "PRJNA607507"
    ).select(
        "study_accession",
        "reference_doi",
        "sample"
    ).unique()
)

study_accession,reference_doi,sample
str,str,str
"""PRJNA607507""","""https://doi.org/10.1101/gad.346106.120""","""DLD-1"""


In [6]:
# PRJNA915030 is GEO GSE221579. The series sequences Jurkat T cells,
# and NCBI links it to doi:10.3390/ijms24021351, not the Nature Kacme paper.
metadata_rnaseq_all = metadata_rnaseq_all.with_columns(
    pl.when(
        (pl.col("study_accession") == "PRJNA915030")
        & (pl.col("sample") == "T lymphocyte")
    )
    .then(pl.lit("Jurkat T cells"))
    .otherwise(pl.col("sample"))
    .alias("sample"),
    pl.when(
        (pl.col("study_accession") == "PRJNA915030")
        & (pl.col("reference_doi") == "https://doi.org/10.1038/s41586-023-06565-9")
    )
    .then(pl.lit("https://doi.org/10.3390/ijms24021351"))
    .otherwise(pl.col("reference_doi"))
    .alias("reference_doi"),
)

metadata_rnaseq_all_sample_list = metadata_rnaseq_all.select(
    "study_accession",
    "reference_doi",
    "sample"
).unique().sort(
    by="sample",
    descending=False
)

display(
    metadata_rnaseq_all.filter(
        pl.col("study_accession") == "PRJNA915030"
    ).select(
        "study_accession",
        "reference_doi",
        "sample"
    ).unique()
)

study_accession,reference_doi,sample
str,str,str
"""PRJNA915030""","""https://doi.org/10.3390/ijms24021351""","""Jurkat T cells"""


In [7]:
# PRJEB44754 stored the culture category "primary cell line".
# The supplementary methods isolate human mesenchymal stem cells from bone marrow.
metadata_rnaseq_all = metadata_rnaseq_all.with_columns(
    pl.when(
        (pl.col("study_accession") == "PRJEB44754")
        & (pl.col("sample") == "primary cell line")
    )
    .then(pl.lit("human mesenchymal stem cells"))
    .otherwise(pl.col("sample"))
    .alias("sample"),
)

metadata_rnaseq_all_sample_list = metadata_rnaseq_all.select(
    "study_accession",
    "reference_doi",
    "sample"
).unique().sort(
    by="sample",
    descending=False
)

display(
    metadata_rnaseq_all.filter(
        pl.col("study_accession") == "PRJEB44754"
    ).select(
        "study_accession",
        "reference_doi",
        "sample"
    ).unique()
)

study_accession,reference_doi,sample
str,str,str
"""PRJEB44754""","""https://doi.org/10.1073/pnas.2210745120""","""human mesenchymal stem cells"""


In [8]:
# PRJNA813310 is GEO GSE197993, a subseries of GSE197995.
# Cugusi et al. deposit that series under doi:10.1128/mcb.00173-22.
# The stored DOI belongs to a different Molecular Cell paper.
metadata_rnaseq_all = metadata_rnaseq_all.with_columns(
    pl.when(
        (pl.col("study_accession") == "PRJNA813310")
        & (pl.col("reference_doi") == "https://doi.org/10.1016/j.molcel.2022.01.007")
    )
    .then(pl.lit("https://doi.org/10.1128/mcb.00173-22"))
    .otherwise(pl.col("reference_doi"))
    .alias("reference_doi"),
)

metadata_rnaseq_all_sample_list = metadata_rnaseq_all.select(
    "study_accession",
    "reference_doi",
    "sample"
).unique().sort(
    by="sample",
    descending=False
)

display(
    metadata_rnaseq_all.filter(
        pl.col("study_accession") == "PRJNA813310"
    ).select(
        "study_accession",
        "reference_doi",
        "sample"
    ).unique()
)


study_accession,reference_doi,sample
str,str,str
"""PRJNA813310""","""https://doi.org/10.1128/mcb.00173-22""","""MRC5-VA"""


In [9]:
# Same DOI correction as the salmon table above.
# This frame is read from the unfiltered TSV, which still has the Molecular Cell DOI.
metadata_rnaseq_all = metadata_rnaseq_all.with_columns(
    pl.when(
        (pl.col("study_accession") == "PRJNA813310")
        & (pl.col("reference_doi") == "https://doi.org/10.1016/j.molcel.2022.01.007")
    )
    .then(pl.lit("https://doi.org/10.1128/mcb.00173-22"))
    .otherwise(pl.col("reference_doi"))
    .alias("reference_doi"),
)

display(
    metadata_rnaseq_all.filter(
        pl.col("study_accession") == "PRJNA813310"
    ).select(
        "study_accession",
        "gse_id",
        "reference_doi"
    ).unique()
)

study_accession,gse_id,reference_doi
str,str,str
"""PRJNA813310""","""GSE197993, GSE197995""","""https://doi.org/10.1128/mcb.00173-22"""


&nbsp;

&nbsp;

&nbsp;

## Sample Mapping

- 先に明示的に対応表を作っておいて、それを統合する方針
- Ontology Lookup Service (OLS) software (https://doi.org/10.5281/zenodo.23237378) を使用して check
- 基本的には、Cell Line Ontology (CLO) を使用するが、EFO, BTO, CL なども併用する
- マッピングできない場合は、親株のオントロジーを使用

In [10]:
sample_ontology_rows = [
    {
        "sample": "A549", # metadata に記載されている sample 名
        "url": "http://purl.obolibrary.org/obo/CLO_0001601",
        "Biosample term id": "CLO_0001601",
        "Biosample term name": "A549 cell",
        "other term id": "EFO:0001086",
    },

    {
        # 論文中では、HeLa cell と呼称されているので、HeLa cell
        "sample": "Biken-HeLa cell",
        "url": "http://purl.obolibrary.org/obo/CLO_0003684",
        "Biosample term id": "CLO_0003684",
        "Biosample term name": "HeLa cell",
        "other term id": "EFO:0001185"
    },

    {
        "sample": "HeLa",
        "url": "http://purl.obolibrary.org/obo/CLO_0003684",
        "Biosample term id": "CLO_0003684",
        "Biosample term name": "HeLa cell",
        "other term id": "EFO:0001185"
    },

    {
        "sample": "HeLa cells",
        "url": "http://purl.obolibrary.org/obo/CLO_0003684",
        "Biosample term id": "CLO_0003684",
        "Biosample term name": "HeLa cell",
        "other term id": "EFO:0001185"
    },

    {
        # CLO にはなかったので、BTO を使用
        "sample": "CNE2",
        "url": "http://purl.obolibrary.org/obo/BTO_0005250",
        "Biosample term id": "BTO_0005250",
        "Biosample term name": "CNE-2 cell",
        "other term id": ""
    },

    {
        "sample": "DLD-1",
        "url": "http://purl.obolibrary.org/obo/CLO_0002785",
        "Biosample term id": "CLO_0002785",
        "Biosample term name": "DLD-1 cell",
        "other term id": "EFO:0006389"
    },

    {
        "sample": "FaDu",
        "url": "http://purl.obolibrary.org/obo/CLO_0002993",
        "Biosample term id": "CLO_0002993",
        "Biosample term name": "FaDu cell",
        "other term id": "EFO:0002178"
    },

    {
        # CLO ではヒットしなかったので、EFO を使用
        "sample": "HAP1",
        "url": "http://www.ebi.ac.uk/efo/EFO_0007598",
        "Biosample term id": "EFO_0007598",
        "Biosample term name": "HAP-1",
        "other term id": ""
    },

    {
        # CLO ではヒットしなかったので、EFO を使用
        "sample": "HAP1 cell",
        "url": "http://www.ebi.ac.uk/efo/EFO_0007598",
        "Biosample term id": "EFO_0007598",
        "Biosample term name": "HAP-1",
        "other term id": ""
    },

    {
        # 
        "sample": "HCT116",
        "url": "http://purl.obolibrary.org/obo/CLO_0003665",
        "Biosample term id": "CLO_0003665",
        "Biosample term name": "HCT 116 cell",
        "other term id": "EFO:0002824"
    },

    {
        "sample": "colorectal cancer cell line HCT116",
        "url": "http://purl.obolibrary.org/obo/CLO_0003665",
        "Biosample term id": "CLO_0003665",
        "Biosample term name": "HCT 116 cell",
        "other term id": "EFO:0002824"
    },

    {
        "sample": "HCT116RmAC",
        "url": "http://purl.obolibrary.org/obo/CLO_0003665",
        "Biosample term id": "CLO_0003665",
        "Biosample term name": "HCT 116 cell",
        "other term id": "EFO:0002824"
    },

    {
        "sample": "HEK293 cells",
        "url": "http://purl.obolibrary.org/obo/CLO_0001230",
        "Biosample term id": "CLO_0001230",
        "Biosample term name": "HEK 293 cell",
        "other term id": "EFO:0001182"
    },

    {
        # ENCODE では、EFO:0001184 にマッピングされていた
        "sample": "HEK293T",
        "url": "http://purl.obolibrary.org/obo/CLO_0037372",
        "Biosample term id": "CLO_0037372",
        "Biosample term name": "HEK293T cell",
        "other term id": "EFO:0001184"
    },

    {
        # CLO ではヒットしなかったので、EFO を使用
        "sample": "HUVEC",
        "url": "http://www.ebi.ac.uk/efo/EFO_0002451",
        "Biosample term id": "EFO_0002451",
        "Biosample term name": "HUVEC",
        "other term id": "BTO:0001949"
    },

    {
        # CLO , EFO ではヒットしなかったので、BTO を使用
        "sample": "HaCaT",
        "url": "http://purl.obolibrary.org/obo/BTO_0000552",
        "Biosample term id": "BTO_0000552",
        "Biosample term name": "HaCaT cell",
        "other term id": ""
    },

    {
        # ENCODE では、EFO:0001187 にマッピングされていた
        "sample": "HepG2 cells",
        "url": "http://purl.obolibrary.org/obo/CLO_0003704",
        "Biosample term id": "CLO_0003704",
        "Biosample term name": "Hep G2 cell",
        "other term id": "EFO:0001187"
    },

    {
        # ENCODE では、EFO:0002067 にマッピングされていた
        "sample": "K562",
        "url": "http://www.ebi.ac.uk/efo/EFO_0002067",
        "Biosample term id": "EFO_0002067",
        "Biosample term name": "K562",
        "other term id": ""
    },

    {
        # ENCODE では、EFO:0001203 にマッピングされていたため、そちらを記載
        "sample": "MCF7",
        "url": "http://purl.obolibrary.org/obo/CLO_0007606",
        "Biosample term id": "CLO_0007606",
        "Biosample term name": "MCF7 cell",
        "other term id": "EFO:0001203"
    },

    {
        # Parental MDA-MB-231. sgHSF1 and sgHSF2 in this study are knockouts of this line.
        "sample": "MDAMB231",
        "url": "http://purl.obolibrary.org/obo/CLO_0007634",
        "Biosample term id": "CLO_0007634",
        "Biosample term name": "MDA-MB-231 cell",
        "other term id": "EFO:0001209",
    },

    {
        # CLO ではヒットしなかったので、BTO を使用
        "sample": "MHCC-97H",
        "url": "http://purl.obolibrary.org/obo/BTO_0004169",
        "Biosample term id": "BTO_0004169",
        "Biosample term name": "MHCC97-H cell",
        "other term id": "",
    },

    {
        "sample": "MM.1S",
        "url": "http://purl.obolibrary.org/obo/CLO_0037203",
        "Biosample term id": "CLO_0037203",
        "Biosample term name": "MM.1S cell",
        "other term id": "EFO:0005724", # ENCODE でもこちらにマッピングされていた
    },

    {
        # MRC5-VA は、CLO ではヒットしなかったので、親株 の MRC-5 を使用
        # Crick Cell Services MRC5VA: SV40-transformed MRC-5. No class is named MRC5-VA.
        # MRC-5V1 (EFO_0022549) and MRC-5 SV2 (CLO_0007868) are separate SV40 derivatives.
        "sample": "MRC5-VA",
        "url": "http://purl.obolibrary.org/obo/CLO_0007865",
        "Biosample term id": "CLO_0007865",
        "Biosample term name": "MRC-5 cell",
        "other term id": "EFO:0002835"
    },

    {
        "sample": "Nalm6",
        "url": "http://purl.obolibrary.org/obo/CLO_0007938",
        "Biosample term id": "CLO_0007938",
        "Biosample term name": "Nalm6 cell",
        "other term id": "EFO:0002245"
    },

    {
        # CLO ではヒットしなかったので、EFO を使用
        "sample": "OCI-Ly1",
        "url": "http://www.ebi.ac.uk/efo/EFO_0005907",
        "Biosample term id": "EFO_0005907",
        "Biosample term name": "OCI-LY1",
        "other term id": ""
    },

    {
        "sample": "OVCAR8",
        "url": "http://www.ebi.ac.uk/efo/EFO_0005444",
        "Biosample term id": "EFO_0005444",
        "Biosample term name": "OVCAR8",
        "other term id": ""
    },

    {
        # CLO ではヒットしなかったので、EFO を使用
        "sample": "PC3M",
        "url": "http://www.ebi.ac.uk/efo/EFO_0022442",
        "Biosample term id": "EFO_0022442",
        "Biosample term name": "PC-3M cell",
        "other term id": ""
    },

    {
        # Fetal telencephalon progenitors (UBERON:0001893) differentiated to neurons.
        # CL:0012001 is the broader forebrain neuron class.
        # 完全一致ではないものの、こちらを使用し、EFO ではヒットしないと思われるので、空の状態にしておく
        "sample": "Primary neurons",
        "url": "http://purl.obolibrary.org/obo/CL_0000540",
        "Biosample term id": "CL_0000540",
        "Biosample term name": "neuron",
        "other term id": "",
    },

    {
        # JCRB 細胞バンク (https://cellbank.nibn.go.jp/~cellbank/cgi-bin/search_res_det.cgi?ID=347) から情報を入手
        # Cellosaurus(ExPASy) の CVCL_1675 の 「Cell line databases/resources」ページから、対応関係を確認
        # alternative label に "SAS" があるので、こちらを使用
        "sample": "SAS",
        "url": "http://purl.obolibrary.org/obo/CLO_0051568",
        "Biosample term id": "CLO_0051568",
        "Biosample term name": "RCB1974 cell",
        "other term id": ""
    },

    {
        "sample": "SW620",
        "url": "http://purl.obolibrary.org/obo/CLO_0009221",
        "Biosample term id": "CLO_0009221",
        "Biosample term name": "SW620 cell",
        "other term id": "EFO:0002368"
    },

    {
        "sample": "Jurkat T cells",
        "url": "http://purl.obolibrary.org/obo/CLO_0007043",
        "Biosample term id": "CLO_0007043",
        "Biosample term name": "JURKAT cell",
        "other term id": "EFO:0002796"
    },

    {
        # JCRB 細胞バンク (https://cellbank.nibn.go.jp/~cellbank/cgi-bin/search_res_det.cgi?ID=347) から情報を入手
        # Cellosaurus(ExPASy) の CVCL_0006 の 「Cell line databases/resources」ページから、対応関係を確認
        # alternative label に "THP-1" があるので、こちらを使用
        "sample": "THP-1",
        "url": "http://purl.obolibrary.org/obo/CLO_0050999",
        "Biosample term id": "CLO_0050999",
        "Biosample term name": "RCB1189 cell",
        "other term id": ""
    },

    {
        # Cellosaurus のページ (https://www.cellosaurus.org/CVCL_M095) を見ても、CLO へのマッピングがない (2026-10-13)
        # ただし、Cell Ontology には対応があるので、これを採用
        "sample": "TIGK",
        "url": "http://purl.obolibrary.org/obo/CL_0000312",
        "Biosample term id": "CL_0000312",
        "Biosample term name": "keratinocyte",
        "other term id": ""
    },

    {
        "sample": "U2OS",
        "url": "http://purl.obolibrary.org/obo/CLO_0009454",
        "Biosample term id": "CLO_0009454",
        "Biosample term name": "U-2 OS cell",
        "other term id": "EFO:0002869"
    },

    {
        # Cellosaurus のページ (https://www.cellosaurus.org/CVCL_0001) を見ても、CLO へのマッピングがない (2026-10-13)
        # ただし、Cell Ontology には対応があるので (https://www.cellosaurus.org/CVCL_N806)、これを採用しようと思ったが、MRC5-VA と同じアプローチ、つまり 親株 (WI-38) へのマッピング方式に変更
        "sample": "WI-38-hTERT",
        "url": "http://purl.obolibrary.org/obo/CLO_0009591",
        "Biosample term id": "CLO_0009591",
        "Biosample term name": "WI 38 cell",
        "other term id": ""
    },

    {
        "sample": "WI38",
        "url": "http://purl.obolibrary.org/obo/CLO_0009591",
        "Biosample term id": "CLO_0009591",
        "Biosample term name": "WI 38 cell",
        "other term id": ""
    },

    {
        "sample": "WI38 lung fibroblast cells",
        "url": "http://purl.obolibrary.org/obo/CLO_0009591",
        "Biosample term id": "CLO_0009591",
        "Biosample term name": "WI 38 cell",
        "other term id": ""
    },

    {
        # Cellosaurus のページ (https://www.cellosaurus.org/CVCL_3768) から CLO へのマッピングを確認
        "sample": "Wild-type OV90 cells",
        "url": "http://purl.obolibrary.org/obo/CLO_0008276",
        "Biosample term id": "CLO_0008276",
        "Biosample term name": "OV-90 cell",
        "other term id": ""
    },

    {
        # 論文を読んでみると、患者さんから採取したサンプルのため、UBERON を使用する
        "sample": "cartilage",
        "url": "http://purl.obolibrary.org/obo/UBERON_0010996",
        "Biosample term id": "UBERON_0010996",
        "Biosample term name": "articular cartilage of joint",
        "other term id": ""
    },

    {
        # オルガノイドのサンプルは、UBERON でも該当がなさそう (分化した部位にあたる語彙としては、おそらく UBERON:0000956 が適切)
        # NCI Thesaurus OBO Edition (ver 26.02d) に Organoid があるので、これを使用
        # other term id には、 UBERON:0000956 を記載
        "sample": "cortical brain organoids",
        "url": "http://purl.obolibrary.org/obo/NCIT_C172259",
        "Biosample term id": "NCIT_C172259",
        "Biosample term name": "Organoid",
        "other term id": "UBERON:0000956"
    },

    {
        # Cellosaurus のページ (https://www.cellosaurus.org/CVCL_4388) を見ると、EFO のエントリーに対応づいていることを発見 (2026-10-13)
        # この EFO のエントリーを採用する
        "sample": "hTERT-RPE1",
        "url": "http://www.ebi.ac.uk/efo/EFO_0022539",
        "Biosample term id": "EFO_0022539",
        "Biosample term name": "hTERT-RPE1",
        "other term id": ""
    },

    {
        # hESC から分化させたサンプルであることに注意
        "sample": "human cardiomyocyte (CMC)",
        "url": "http://purl.obolibrary.org/obo/CL_0000746",
        "Biosample term id": "CL_0000746",
        "Biosample term name": "cardiac muscle cell",
        "other term id": ""
    },

    {
        # CLO でも ヒットがあり、alternative label も hESC のため、こちらを採用
        "sample": "human embryonic stem cell (hESC)",
        "url": "http://purl.obolibrary.org/obo/CLO_0037280",
        "Biosample term id": "CLO_0037280",
        "Biosample term name": "human embryonic stem cell line cell",
        "other term id": ""
    },

    {
        # CL から CLO に import されているので、こちらを採用
        "sample": "human mesenchymal stem cell (hMSC)",
        "url": "http://purl.obolibrary.org/obo/CL_0000134",
        "Biosample term id": "CL_0000134",
        "Biosample term name": "mesenchymal stem cell",
        "other term id": ""
    },

    {
        # CL から CLO に import されているので、こちらを採用
        "sample": "human neural stem cell (hNSC)",
        "url": "http://purl.obolibrary.org/obo/CL_0000047",
        "Biosample term id": "CL_0000047",
        "Biosample term name": "neural stem cell",
        "other term id": ""
    },

    {
        # CL から CLO に import されているので、こちらを採用
        "sample": "human neuron (hNeuron)",
        "url": "http://purl.obolibrary.org/obo/CL_0000540",
        "Biosample term id": "CL_0000540",
        "Biosample term name": "neuron",
        "other term id": ""
    },

    {
        # CLO ではヒットせず、CL でヒット
        # Exact Synonyms が vascular endothelial cell となっているので、こちらを採用
        "sample": "human vascular endothelial cell (hVEC)",
        "url": "http://purl.obolibrary.org/obo/CL_0002139",
        "Biosample term id": "CL_0002139",
        "Biosample term name": "endothelial cell of vascular tree",
        "other term id": ""
    },

    {
        "sample": "human vascular smooth muscle cell (hVSMC)",
        "url": "http://purl.obolibrary.org/obo/CL_0000359",
        "Biosample term id": "CL_0000359",
        "Biosample term name": "vascular associated smooth muscle cell",
        "other term id": ""
    },

    {
        # Day-50 telencephalic aggregate from fibroblast-derived iPSC lines C1 and C2.
        # 分化してから 50 日後のサンプルであるので、neuron にマッピング
        # 分化する前の状態は、CLO:0037308 にマッピングされている
        "sample": "neuronal aggregates (day 50) derived from induced pluripotent stem cells",
        "url": "http://purl.obolibrary.org/obo/CL_0000540",
        "Biosample term id": "CL_0000540",
        "Biosample term name": "neuron",
        "other term id": "CLO:0037308",
    },

    {
        # Supplementary Materials and methods では, "Human mesenchymal stem cells (hMSCs) were isolated from bone marrow using established methodology" と記載されている
        # CLO ではヒットせず、CL でヒットしたため、こちらを採用
        "sample": "human mesenchymal stem cells",
        "url": "http://purl.obolibrary.org/obo/CL_0002540",
        "Biosample term id": "CL_0002540",
        "Biosample term name": "mesenchymal stem cell of the bone marrow",
        "other term id": ""
    }

]

In [11]:
sample_ontology = pl.DataFrame(sample_ontology_rows).unique()

metadata_rnaseq_all_ontology = metadata_rnaseq_all.join(
    sample_ontology,
    on="sample",
    how="inner", # ここでは inner で結合しておく
    validate="m:1",
)

print(metadata_rnaseq_all_ontology.columns)

pl.Config.set_tbl_cols(200)
pl.Config.set_tbl_rows(2)
display(metadata_rnaseq_all_ontology)

['sample_heat_id', 'sample_control_id', 'sample_name', 'sample_title', 'experiment_accession', 'experiment_alias', 'secondary_sample_accession', 'study_accession', 'first_public', 'study_alias', 'instrument_platform', 'instrument_model', 'library_source', 'library_layout', 'sample_molecule_ch1', 'sample', 'stress treatment temperature', 'stress treatment time', 'control treatment temperature', 'rna_library', 'normalized_label', 'item_type', 'reference_doi', 'strandness', 'strandness_evidence', 'expected_format_heat', 'strand_mapping_bias_heat', 'expected_format_control', 'strand_mapping_bias_control', 'format_pair_comparison', 'expected_library_strandedness', 'sample_geo_accession', 'sample_status', 'sample_submission_date', 'sample_last_update_date', 'sample_type', 'sample_channel_count', 'sample_source_name_ch1', 'sample_organism_ch1', 'sample_taxid_ch1', 'sample_description', 'sample_platform_id', 'sample_contact_name', 'sample_contact_institute', 'sample_contact_address', 'sample_c

sample_heat_id,sample_control_id,sample_name,sample_title,experiment_accession,experiment_alias,secondary_sample_accession,study_accession,first_public,study_alias,instrument_platform,instrument_model,library_source,library_layout,sample_molecule_ch1,sample,stress treatment temperature,stress treatment time,control treatment temperature,rna_library,normalized_label,item_type,reference_doi,strandness,strandness_evidence,expected_format_heat,strand_mapping_bias_heat,expected_format_control,strand_mapping_bias_control,format_pair_comparison,expected_library_strandedness,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,cell_type,genotype,treatment,mapping,expression_estimation,deg_analysis,assembly,supplementary_files_format_and_content,sample_accession,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,country,ncbi_reporting_standard,sample_description_right,secondary_study_accession,read_count,study_title,sample_title_right,description,cell_type_right,base_count,library_strategy,status,sample_alias,run_alias,library_construction_protocol,collection_date,cell_line_right,library_name,first_created,sample_data_processing,sample_contact_email,tissue,biological_replicates,tissue_type,protocol,organism,read_type,data_source,srr,srx,sample_contact_department,passage,phenotype,replicate,genome_build,center_name,sample_contact_laboratory,sample_supplementary_file_2,sample_supplementary_file_3,disease,stress,sample_contact_phone,tumor,sex,age,sample_treatment_protocol_ch1,sample_group,cell_model,sample_extract_protocol_ch1,time,library_strategy_right,sample_growth_protocol_ch1,sirna,condition,tissue_cells,antibody,batch,column,library_name_right,sample_characteristics_ch1,centrifuge,strain,fraction,enzyme_chemical_treatment,rt_condition,strain_right,treatment_conditions,molecule,isolate,isolate_right,dosage,experiment,timepoint,vector,sirna_sgrna,sra_file_role,fastq_file_role,tag,scientific_name,tax_id,accession,collection_date_end,collection_date_start,submitted_galaxy,submitted_ftp,submitted_aspera,submitted_md5,checklist,submitted_bytes,nominal_sdev,broker_name,datahub,nominal_length,submitted_read_type,submitted_format,bam_ftp,bam_bytes,dev_stage,bam_aspera,bam_galaxy,bam_file_role,submitted_file_role,recovery treatment temperature,recovery treatment time,sample_series_id,biosample,sra,fastq_galaxy,fastq_ftp,fastq_aspera,tax_lineage,Read length for mate1(bp),Read length for mate 2(bp),Individual Accession,Biomaterial Provider,url,Biosample term id,Biosample term name,other term id
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,str,f64,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR27681984""","""SRR27681986""","""wi-38-htert_40°c_rna-seq_rep1 vs wi-38-htert_37°c_rna-seq_rep1""","""WI-38-hTERT, 40°C, RNA-seq, rep1 vs WI-38-hTERT, 

&nbsp;

&nbsp;

&nbsp;

## Genotype Information Process

- genotype の 情報も名寄せしておくことを忘れていた この際なのでマッピングしてしまおう
- DOI の間違いを見つけたので、事前にチェック

&nbsp;

&nbsp;

## For Genotype Mapping

- `with_columns` で、標準化を進めていく
- 既に、サンプルに対してマッピングした結果を持つデータフレームに更に追加していく方式にする

In [12]:
rnaseq_all_genotype_modified = metadata_rnaseq_all_ontology.with_columns(
    pl.coalesce("genotype", "sirna", "sirna_sgrna").alias("genotype_raw"),
    # Filled later, one study at a time. Null means that pair is not normalized yet.
    pl.lit(None, dtype=pl.String).alias("genotype_normalization"),
).sort(
    by=["genotype", "study_accession"],
    descending=True
).with_columns(
    # One column name can appear only once inside a single with_columns.
    # Later rules are extra when().then() links. The first match wins.
    # 1. Project ごとに wild type (WT) だと判断した場合 (これがおそらく一番多いケース)
    pl.when(
        (pl.col("study_accession") == "PRJNA985730")
        | (pl.col("study_accession") == "PRJCA005446")
        | (pl.col("study_accession") == "PRJDB3999")
        | (pl.col("study_accession") == "PRJNA1014403")
        | (pl.col("study_accession") == "PRJNA1113656")
        | (pl.col("study_accession") == "PRJNA1169616")
        | (pl.col("study_accession") == "PRJNA1172720")
        | (pl.col("study_accession") == "PRJNA1221942")
        | (pl.col("study_accession") == "PRJNA232603")
        | (pl.col("study_accession") == "PRJNA316856")
        | (pl.col("study_accession") == "PRJNA391584") # ML00754792 という薬剤を使用しているが、genotype としては WT となる
        | (pl.col("study_accession") == "PRJNA475962")
        | (pl.col("study_accession") == "PRJNA510428") # よく見たら 4sU-labelling を行っていた
        | (pl.col("study_accession") == "PRJNA512268")
        | (pl.col("study_accession") == "PRJNA512837") # young と Senescent の違いはある
        | (pl.col("study_accession") == "PRJNA541915") 
        | (pl.col("study_accession") == "PRJNA548108")
        | (pl.col("study_accession") == "PRJNA565925")
        | (pl.col("study_accession") == "PRJNA630173")
        | (pl.col("study_accession") == "PRJNA634094")
        | (pl.col("study_accession") == "PRJNA692199")
        | (pl.col("study_accession") == "PRJNA747271")
        | (pl.col("study_accession") == "PRJNA783644")
        | (pl.col("study_accession") == "PRJNA815310")
        | (pl.col("study_accession") == "PRJNA956578")
        | (pl.col("study_accession") == "PRJNA915030")
        | (pl.col("study_accession") == "PRJNA944942")
        | (pl.col("study_accession") == "PRJNA949515")
    )
    .then(pl.lit("wild type (WT)"))


    # 2. genotype_raw が既に WT のペア (project は問わない)
    # PRJNA799044 stores the MDA-MB-231 wild-type arm as the string "None".
    .when(
        (pl.col("genotype_raw") == "WT")
        | (pl.col("genotype_raw") == "wild type")
        | (pl.col("genotype_raw") == "WT cells")
        | (
            (pl.col("study_accession") == "PRJNA799044")
            & (pl.col("genotype_raw") == "None")
        )
    )
    .then(pl.lit("wild type (WT)"))


    # 3. HIKESHI KO
    .when(
        (pl.col("genotype_raw") == "HIKESHI knockout")
    )
    .then(pl.lit("knockout (KO) (target: HIKESHI)"))

    # 4. HIKESHI KD
    .when(
        (pl.col("genotype_raw") == "HIKESHI knockdown")
    )
    .then(pl.lit("knockdown (KD) (target: HIKESHI)"))


    # PRJNA476608 U2OS knockouts. The metadata spells factor as "farctor".
    # The wild type pairs in this study are already covered by rule 2.
    # PRJNA799044 sgHSF1.1 and sgHSF2.1 are Brunello guide ids for the same knockouts.
    .when(
        (pl.col("genotype_raw") == "heat shock farctor 1 KO")
        | (pl.col("genotype_raw") == "sgHSF1.1")
    )
    .then(pl.lit("knockout (KO) (target: HSF1)"))
    .when(
        (pl.col("genotype_raw") == "heat shock farctor 2 KO")
        | (pl.col("genotype_raw") == "sgHSF2.1")
    )
    .then(pl.lit("knockout (KO) (target: HSF2)"))


    # 5. Overexpression
    .when(
        (pl.col("genotype_raw") == "CLK2 (WT)")
    )
    .then(pl.lit("overexpression (target: CLK2 (WT))"))

    .when(
        (pl.col("genotype_raw") == "Control") & (pl.col("study_accession") == "PRJNA1037521")
    )
    .then(pl.lit("overexpression (target: GFP)"))


    # 6. PRJNA1080614 shRNA knockdowns in HCT116, including Abcf2, Acaca, and Acacb.
    # Scrambled control is the paper's negative control shScr.
    .when(
        (pl.col("study_accession") == "PRJNA1080614")
        & pl.col("genotype_raw").str.ends_with(" knockdown")
    )
    .then(
        pl.format(
            "knockdown (KD) (target: {})",
            pl.col("genotype_raw").str.replace(" knockdown", "", literal=True),
        )
    )
    .when(
        (pl.col("study_accession") == "PRJNA1080614")
        & pl.col("genotype_raw").is_in(["Abcf2", "Acaca", "Acacb"])
    )
    .then(
        pl.format(
            "knockdown (KD) (target: {})",
            pl.col("genotype_raw"),
        )
    )
    .when(
        ((pl.col("study_accession") == "PRJNA1080614") & (pl.col("genotype_raw") == "Scrambled control"))
        | (pl.col("genotype_raw") == "siNT")
        | (pl.col("genotype_raw") == "non-targeting siRNA")

    )
    .then(pl.lit("knockdown (KD) (target: non-targeting)"))


    # 7. PRJNA1134123 shares one untreated GFP control across three heat arms.
    # Only the Control arm is the same genotype on both sides.
    # ペアの作り方に注意 (genotype まで一致しているペアではないことに注意！)
    .when(
        (pl.col("study_accession") == "PRJNA1134123")
        & (pl.col("genotype_raw") == "Control")
    )
    .then(pl.lit("overexpression (target: GFP)"))
    .when(
        (pl.col("study_accession") == "PRJNA1134123")
        & (pl.col("genotype_raw") == "CLK2 (WT) vs Control")
    )
    .then(pl.lit("overexpression (target: CLK2 (WT)) vs overexpression (target: GFP)"))
    .when(
        (pl.col("study_accession") == "PRJNA1134123")
        & (pl.col("genotype_raw") == "CLK2 knockdown vs Control")
    )
    .then(pl.lit("knockdown (KD) (target: CLK2) vs overexpression (target: GFP)"))


    # 8. PRJNA297109. The RNAi text is in treatment, not genotype_raw.
    # Heat-only pairs are unmodified WI-38. BRD4 siRNA is on both sides.
    .when(
        (pl.col("study_accession") == "PRJNA297109")
        & pl.col("treatment").str.starts_with("To induce heat stress")
    )
    .then(pl.lit("wild type (WT)"))
    .when(
        (pl.col("study_accession") == "PRJNA297109")
        & pl.col("treatment").str.contains("RNA interference targeting BRD4", literal=True)
    )
    .then(pl.lit("knockdown (KD) (target: BRD4)"))


    # 9. PRJNA607507. Auxin induces degradation of the AID-tagged protein.
    # Water leaves the same tagged protein in place. genotype_raw is only the cell line.
    .when(
        (pl.col("study_accession") == "PRJNA607507")
        & (pl.col("genotype_raw") == "BRD4-AID OsTIR1")
        & pl.col("sample_title").str.starts_with("2hAux_")
    )
    .then(pl.lit("degron (AID) (target: BRD4)"))
    .when(
        (pl.col("study_accession") == "PRJNA607507")
        & (pl.col("genotype_raw") == "BRD4-AID OsTIR1")
        & pl.col("sample_title").str.starts_with("2hH2O_")
    )
    .then(pl.lit("degron (AID), not induced (target: BRD4)"))
    .when(
        (pl.col("study_accession") == "PRJNA607507")
        & (pl.col("genotype_raw") == "AFF1/4-AID OsTIR1")
        & pl.col("sample_title").str.starts_with("2hAux_")
    )
    .then(pl.lit("degron (AID) (target: AFF1, AFF4)"))
    .when(
        (pl.col("study_accession") == "PRJNA607507")
        & (pl.col("genotype_raw") == "AFF1/4-AID OsTIR1")
        & pl.col("sample_title").str.starts_with("2hH2O_")
    )
    .then(pl.lit("degron (AID), not induced (target: AFF1, AFF4)"))


    # 10. PRJNA670565 MCF7. Two HSF1 knockout clones share one label.
    # wild type is already covered by rule 2.
    .when(
        (pl.col("study_accession") == "PRJNA670565")
        & pl.col("genotype_raw").is_in([
            "CRISPR/Cas9 HSF1 knockout 1",
            "CRISPR/Cas9 HSF1 knockout 2",
        ])
    )
    .then(pl.lit("knockout (KO) (target: HSF1)"))
    .when(
        (pl.col("study_accession") == "PRJNA670565")
        & (pl.col("genotype_raw") == "shHSF1")
    )
    .then(pl.lit("knockdown (KD) (target: HSF1)"))
    .when(
        (pl.col("study_accession") == "PRJNA670565")
        & (pl.col("genotype_raw") == "shRNA control")
    )
    .then(pl.lit("knockdown (KD) (target: non-targeting)"))
    .when(
        (pl.col("study_accession") == "PRJNA670565")
        & (pl.col("genotype_raw") == "CRISPR/Cas9 control")
    )
    .then(pl.lit("knockout (KO) (target: non-targeting)"))


    # 11. PRJNA692181. SW620. The siRNA is in treatment, and both sides of a pair share it.
    .when(
        (pl.col("study_accession") == "PRJNA692181")
        & (pl.col("treatment") == "Scrambled siRNA")
    )
    .then(pl.lit("knockdown (KD) (target: non-targeting)"))
    .when(
        (pl.col("study_accession") == "PRJNA692181")
        & (pl.col("treatment") == "Lonp1 siRNA")
    )
    .then(pl.lit("knockdown (KD) (target: LONP1)"))


    # 12. PRJNA758423 HCT116 RAD21-mAID. Both sides of a pair share the degrader.
    # MZ1+IAA is its own prefix, so it does not match the MZ1-only rule.
    .when(
        (pl.col("study_accession") == "PRJNA758423")
        & pl.col("treatment").str.starts_with("DMSO under heat shock")
    )
    .then(pl.lit("degron (AID), not induced (target: RAD21)"))
    .when(
        (pl.col("study_accession") == "PRJNA758423")
        & pl.col("treatment").str.starts_with("IAA under heat shock")
    )
    .then(pl.lit("degron (AID) (target: RAD21)"))
    .when(
        (pl.col("study_accession") == "PRJNA758423")
        & pl.col("treatment").str.starts_with("MZ1 under heat shock")
    )
    .then(pl.lit("degron (AID), not induced (target: RAD21) + PROTAC (target: BRD4)"))
    .when(
        (pl.col("study_accession") == "PRJNA758423")
        & pl.col("treatment").str.starts_with("MZ1+IAA under heat shock")
    )
    .then(pl.lit("degron (AID) (target: RAD21) + PROTAC (target: BRD4)"))

    # 13. PRJNA813310
    .when(
        (pl.col("genotype_raw") == "RPRD1B siRNA")
    )
    .then(pl.lit("knockdown (KD) (target: RPRD1B)"))


    # 14. PRJNA918302 HEK293T. Both sides of a pair share the siRNA.
    # E2combo is a pool against the E2 ubiquitin-conjugating enzymes.
    .when(
        (pl.col("study_accession") == "PRJNA918302")
        & pl.col("treatment").str.starts_with("control siRNAs")
    )
    .then(pl.lit("knockdown (KD) (target: non-targeting)"))
    .when(
        (pl.col("study_accession") == "PRJNA918302")
        & pl.col("treatment").str.starts_with("E2combo siRNAs")
    )
    .then(pl.lit("knockdown (KD) (target: E2s)"))


    .otherwise(pl.col("genotype_normalization"))
    .alias("genotype_normalization")
)

In [13]:
display(rnaseq_all_genotype_modified)

sample_heat_id,sample_control_id,sample_name,sample_title,experiment_accession,experiment_alias,secondary_sample_accession,study_accession,first_public,study_alias,instrument_platform,instrument_model,library_source,library_layout,sample_molecule_ch1,sample,stress treatment temperature,stress treatment time,control treatment temperature,rna_library,normalized_label,item_type,reference_doi,strandness,strandness_evidence,expected_format_heat,strand_mapping_bias_heat,expected_format_control,strand_mapping_bias_control,format_pair_comparison,expected_library_strandedness,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,cell_type,genotype,treatment,mapping,expression_estimation,deg_analysis,assembly,supplementary_files_format_and_content,sample_accession,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,country,ncbi_reporting_standard,sample_description_right,secondary_study_accession,read_count,study_title,sample_title_right,description,cell_type_right,base_count,library_strategy,status,sample_alias,run_alias,library_construction_protocol,collection_date,cell_line_right,library_name,first_created,sample_data_processing,sample_contact_email,tissue,biological_replicates,tissue_type,protocol,organism,read_type,data_source,srr,srx,sample_contact_department,passage,phenotype,replicate,genome_build,center_name,sample_contact_laboratory,sample_supplementary_file_2,sample_supplementary_file_3,disease,stress,sample_contact_phone,tumor,sex,age,sample_treatment_protocol_ch1,sample_group,cell_model,sample_extract_protocol_ch1,time,library_strategy_right,sample_growth_protocol_ch1,sirna,condition,tissue_cells,antibody,batch,column,library_name_right,sample_characteristics_ch1,centrifuge,strain,fraction,enzyme_chemical_treatment,rt_condition,strain_right,treatment_conditions,molecule,isolate,isolate_right,dosage,experiment,timepoint,vector,sirna_sgrna,sra_file_role,fastq_file_role,tag,scientific_name,tax_id,accession,collection_date_end,collection_date_start,submitted_galaxy,submitted_ftp,submitted_aspera,submitted_md5,checklist,submitted_bytes,nominal_sdev,broker_name,datahub,nominal_length,submitted_read_type,submitted_format,bam_ftp,bam_bytes,dev_stage,bam_aspera,bam_galaxy,bam_file_role,submitted_file_role,recovery treatment temperature,recovery treatment time,sample_series_id,biosample,sra,fastq_galaxy,fastq_ftp,fastq_aspera,tax_lineage,Read length for mate1(bp),Read length for mate 2(bp),Individual Accession,Biomaterial Provider,url,Biosample term id,Biosample term name,other term id,genotype_raw,genotype_normalization
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,str,f64,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR24973772""","""SRR24973776""","""m3d_heat_1_rna_seq vs m3d_ctrl_1_rna_seq""","""M3D_Heat_1_RNA_seq vs M3D_C

&nbsp;

&nbsp;

&nbsp;

### Classification (2)

- 大まかに分ける

In [ ]:
rnaseq_all_genotype_check = rnaseq_all_genotype_modified.drop(
    "genotype_raw"
).with_columns(
    pl.when(pl.col("genotype_normalization") == "wild type (WT)")
    .then(pl.lit("WT"))
    .when(
        pl.col("genotype_normalization").str.starts_with("knockdown (KD)")
        & ~pl.col("genotype_normalization").str.contains(" vs ", literal=True)
    )
    .then(pl.lit("KD"))
    .when(pl.col("genotype_normalization").str.starts_with("knockout (KO)"))
    .then(pl.lit("KO"))
    .when(pl.col("genotype_normalization").str.starts_with("overexpression"))
    .then(pl.lit("overexpression"))
    .when(pl.col("genotype_normalization").is_not_null())
    .then(pl.lit("Other"))
    .otherwise(pl.lit(None, dtype=pl.String))
    .alias("genotype_class")
)


pl.Config.set_tbl_cols(200)
pl.Config.set_tbl_rows(2)
display(rnaseq_all_genotype_check)

sample_heat_id,sample_control_id,sample_name,sample_title,experiment_accession,experiment_alias,secondary_sample_accession,study_accession,first_public,study_alias,instrument_platform,instrument_model,library_source,library_layout,sample_molecule_ch1,sample,stress treatment temperature,stress treatment time,control treatment temperature,rna_library,normalized_label,item_type,reference_doi,strandness,strandness_evidence,expected_format_heat,strand_mapping_bias_heat,expected_format_control,strand_mapping_bias_control,format_pair_comparison,expected_library_strandedness,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,cell_type,genotype,treatment,mapping,expression_estimation,deg_analysis,assembly,supplementary_files_format_and_content,sample_accession,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,country,ncbi_reporting_standard,sample_description_right,secondary_study_accession,read_count,study_title,sample_title_right,description,cell_type_right,base_count,library_strategy,status,sample_alias,run_alias,library_construction_protocol,collection_date,cell_line_right,library_name,first_created,sample_data_processing,sample_contact_email,tissue,biological_replicates,tissue_type,protocol,organism,read_type,data_source,srr,srx,sample_contact_department,passage,phenotype,replicate,genome_build,center_name,sample_contact_laboratory,sample_supplementary_file_2,sample_supplementary_file_3,disease,stress,sample_contact_phone,tumor,sex,age,sample_treatment_protocol_ch1,sample_group,cell_model,sample_extract_protocol_ch1,time,library_strategy_right,sample_growth_protocol_ch1,sirna,condition,tissue_cells,antibody,batch,column,library_name_right,sample_characteristics_ch1,centrifuge,strain,fraction,enzyme_chemical_treatment,rt_condition,strain_right,treatment_conditions,molecule,isolate,isolate_right,dosage,experiment,timepoint,vector,sirna_sgrna,sra_file_role,fastq_file_role,tag,scientific_name,tax_id,accession,collection_date_end,collection_date_start,submitted_galaxy,submitted_ftp,submitted_aspera,submitted_md5,checklist,submitted_bytes,nominal_sdev,broker_name,datahub,nominal_length,submitted_read_type,submitted_format,bam_ftp,bam_bytes,dev_stage,bam_aspera,bam_galaxy,bam_file_role,submitted_file_role,recovery treatment temperature,recovery treatment time,sample_series_id,biosample,sra,fastq_galaxy,fastq_ftp,fastq_aspera,tax_lineage,Read length for mate1(bp),Read length for mate 2(bp),Individual Accession,Biomaterial Provider,url,Biosample term id,Biosample term name,other term id,genotype_normalization,genotype_class
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,str,f64,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR24973772""","""SRR24973776""","""m3d_heat_1_rna_seq vs m3d_ctrl_1_rna_seq""","""M3D_Heat_1_RNA_seq vs M3D

&nbsp;

&nbsp;

&nbsp;

## Rearranging Columns

- 全てのカラムに値が入っている 「_simple」と、その他のカラムがある 「_all」カラムをここでもう一度作り直す

In [22]:
simple_colmuns_list = [
    "sample_heat_id",
    "sample_control_id",
    # "sample_name",
    "sample_title",
    "experiment_accession",
    "experiment_alias",
    "secondary_sample_accession",
    "study_accession",
    "first_public",
    "study_alias",
    "instrument_platform",
    "instrument_model",
    "library_source",
    "library_layout",
    "sample_molecule_ch1",
    "rna_library", # add
    "normalized_label", # add
    "item_type", # add
    "strandness", # add
    "strandness_evidence", # add
    "expected_format_heat", # add
    "expected_format_control", # add
    "strand_mapping_bias_heat", # add
    "strand_mapping_bias_control", # add
    "format_pair_comparison", # add
    "expected_library_strandedness", # add
    "sample",
    "url", # add
    "Biosample term id", # add
    "Biosample term name", # add
    "other term id", # add
    "genotype_normalization", # add
    "genotype_class", # add
    "stress treatment temperature", # add
    "stress treatment time", # add
    "control treatment temperature", # add
    "reference_doi" # add
]

In [26]:
rnaseq_metadata_all_modified_simple = rnaseq_all_genotype_check.select(
    simple_colmuns_list
)

rnaseq_metadata_all_modified_simple.write_csv(
    "../../Data/metadata/metadata_2610/rnaseq_all_complete/rnaseq_all_metadata_modified_simple_261014.tsv",
    separator="\t"
)

# figshare に upload するための directory にも念のため書き出しておく
rnaseq_metadata_all_modified_simple.write_csv(
    "../../out/figshare/metadata/rnaseq_all_metadata_modified_simple_261014.tsv",
    separator="\t"
)


display(rnaseq_metadata_all_modified_simple)

sample_heat_id,sample_control_id,sample_title,experiment_accession,experiment_alias,secondary_sample_accession,study_accession,first_public,study_alias,instrument_platform,instrument_model,library_source,library_layout,sample_molecule_ch1,rna_library,normalized_label,item_type,strandness,strandness_evidence,expected_format_heat,expected_format_control,strand_mapping_bias_heat,strand_mapping_bias_control,format_pair_comparison,expected_library_strandedness,sample,url,Biosample term id,Biosample term name,other term id,genotype_normalization,genotype_class,stress treatment temperature,stress treatment time,control treatment temperature,reference_doi
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,f64,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR24973772""","""SRR24973776""","""M3D_Heat_1_RNA_seq vs M3D_Ctrl_1_RNA_seq""","""SRX20731424 vs SRX20731421""","""GSM7500881_r1 vs GSM7500878_r1""","""SRS18022362 vs SRS18022360""","""PRJNA985730""","""2023-06-27""","""GSE235351""","""BGISEQ""","""MGISEQ-2000RS""","""TRANSCRIPTOMIC""","""SINGLE""","""total RNA""","""unknown""","""unknown""","""missing""","""unknown""","""""","""U""","""U""",0.501465,0.499687,"""same format""","""unstranded library pairs""","""MHCC-97H""","""http://purl.obolibrary.org/obo/BTO_0004169""","""BTO_0004169""","""MHCC97-H cell""","""""","""wild type (WT)""","""WT""","""37~53 C""","""15 min.""","""37 C (inferred)""","""https://doi.org/10.1016/j.xcrm.2023.101128"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""SRR12926180""","""SRR12926176""","""2hH2O_4h42C_DoubleAID_rep2 vs 2hH2O_4h37C_DoubleAID_rep2""","""SRX9390318 vs SRX9390316""","""GSM4871718 vs GSM4871716""","""SRS7609616 vs SRS7609614""","""PRJNA607507""","""2021-01-17""","""GSE145525""","""ILLUMINA""","""Illumina NovaSeq 6000""","""TRANSCRIPTOMIC""","""SINGLE""","""total RNA""","""NEBNext Ultra II Directional RNA library preparation kit""","""NEBNext Ultra II Directional RNA Library Prep Kit for Illumina""","""library_preparation""","""stranded""","""https://www.neb.com/en/products/e7760-nebnext-ultra-ii-directional-rna-library-prep-kit-for-illumina""","""SR""","""SR""",0.000007,0.000009,"""same format""","""stranded library pairs""","""DLD-1""","""http://purl.obolibrary.org/obo/CLO_0002785""","""CLO_0002785""","""DLD-1 cell""","""EFO:0006389""","""degron (AID), not induced (target: AFF1, AFF4)""","""Other""","""42 C""","""240 min.""","""37 C (inferred)""","""https://doi.org/10.1101/gad.346106.120"""


In [27]:
rnaseq_metadata_all_modified_all = (
    rnaseq_metadata_all_modified_simple
    .join(
        rnaseq_all_genotype_check,
        on=simple_colmuns_list,
        how="inner",
        validate="1:1",
        nulls_equal=True # 念のため
    )
)

rnaseq_metadata_all_modified_all.write_csv(
    "../../Data/metadata/metadata_2610/rnaseq_all_complete/rnaseq_all_metadata_modified_all_261014.tsv",
    separator="\t"
)

# figshare に upload するための directory にも念のため書き出しておく
rnaseq_metadata_all_modified_all.write_csv(
    "../../out/figshare/metadata/rnaseq_all_metadata_modified_all_261014.tsv",
    separator="\t"
)

display(rnaseq_metadata_all_modified_all)

sample_heat_id,sample_control_id,sample_title,experiment_accession,experiment_alias,secondary_sample_accession,study_accession,first_public,study_alias,instrument_platform,instrument_model,library_source,library_layout,sample_molecule_ch1,rna_library,normalized_label,item_type,strandness,strandness_evidence,expected_format_heat,expected_format_control,strand_mapping_bias_heat,strand_mapping_bias_control,format_pair_comparison,expected_library_strandedness,sample,url,Biosample term id,Biosample term name,other term id,genotype_normalization,genotype_class,stress treatment temperature,stress treatment time,control treatment temperature,reference_doi,sample_name,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,cell_type,genotype,treatment,mapping,expression_estimation,deg_analysis,assembly,supplementary_files_format_and_content,sample_accession,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,country,ncbi_reporting_standard,sample_description_right,secondary_study_accession,read_count,study_title,sample_title_right,description,cell_type_right,base_count,library_strategy,status,sample_alias,run_alias,library_construction_protocol,collection_date,cell_line_right,library_name,first_created,sample_data_processing,sample_contact_email,tissue,biological_replicates,tissue_type,protocol,organism,read_type,data_source,srr,srx,sample_contact_department,passage,phenotype,replicate,genome_build,center_name,sample_contact_laboratory,sample_supplementary_file_2,sample_supplementary_file_3,disease,stress,sample_contact_phone,tumor,sex,age,sample_treatment_protocol_ch1,sample_group,cell_model,sample_extract_protocol_ch1,time,library_strategy_right,sample_growth_protocol_ch1,sirna,condition,tissue_cells,antibody,batch,column,library_name_right,sample_characteristics_ch1,centrifuge,strain,fraction,enzyme_chemical_treatment,rt_condition,strain_right,treatment_conditions,molecule,isolate,isolate_right,dosage,experiment,timepoint,vector,sirna_sgrna,sra_file_role,fastq_file_role,tag,scientific_name,tax_id,accession,collection_date_end,collection_date_start,submitted_galaxy,submitted_ftp,submitted_aspera,submitted_md5,checklist,submitted_bytes,nominal_sdev,broker_name,datahub,nominal_length,submitted_read_type,submitted_format,bam_ftp,bam_bytes,dev_stage,bam_aspera,bam_galaxy,bam_file_role,submitted_file_role,recovery treatment temperature,recovery treatment time,sample_series_id,biosample,sra,fastq_galaxy,fastq_ftp,fastq_aspera,tax_lineage,Read length for mate1(bp),Read length for mate 2(bp),Individual Accession,Biomaterial Provider
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR24973772""","""SRR24973776""","""M3D_Heat_1_RNA_seq vs M3D_Ctrl_1_RNA_seq""","""SRX20731424 vs SRX2073142